In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Real crop observation: fixed existing-video diagnostic

Run all reads the six saved OFF/A/B MP4 videos from the FIXED source run
`InversionStateHoldout/inversion_state_holdout_20260918T140243514638Z`.
No generation. Each source gets 129-frame crops at starts 0, 16 and 17: 18 clips.
Crops are saved as lossless RGB MP4 (libx264rgb, CRF0), read back and checked
pixel-for-pixel against the selected source RGB frames. No padding to 181 frames.

Receiver uses actual clip pixels -> VAE normalized mode -> the unchanged
50-step approximate Euler inverse with public prompt/negative/CFG. 129 frames
produce 33 latent slices. Receiver accepts no crop start, true message, writer
noise or time pad. It persists recovered tensors and raw local summaries.

ONLY a separate posthoc oracle stage uses source timing and pads. Start0 maps
nominal shift0; start16 maps shift4; start17 reports BOTH floor4 and ceil5, never
selects the better one. First local latent is kept as a reset boundary and
excluded from core scores. Last local latent remains included as a full group.
Only complete 4/4-slice windows are scored; partial, missing and invalid rows
remain. Nominal grid alignment never guarantees original latent equality.

Fixed denominators: six source videos, 18 correlated clips, 594 local slices;
24 related oracle mappings / 264 core windows. Marked clips: 12, marked maps:
16, marked core grid: 176. Each marked map group has 4 source videos; complete
window denominators for start0/16/17floor/17ceil are 32/32/32/28. OFF ranks are
oracle diagnostics with no true label, not positive detections or a low-FPR test.
Observer on/off is auxiliary, with missing cost1. Different-coverage scores
must not be compared as equally supported. No blind synchronization claim.

Budget: 18 lossless MP4 saves, 18 VAE encodes, 1800 Transformer calls, 900 inverse
updates. Zero generation, zero VAE decode. VAE and Transformer lifetimes remain
separate. Source files are read-only; output is a new directory under
`MyDrive/Video-WM/InversionCropObservation/inversion_crop_observation_<UTC>`.
Failures keep fixed slots, logs and source archive. No mode selection or scan.
Select GPU and Run all; there is no GPU model-name requirement. Only local
CPU/fake/static checks have run, not this pretrained-model experiment.

Source SHA: 8ad796e1bfa5d3f5dc697c3f14f62df0f24f0d29.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata, json, os, signal, subprocess, sys
SOURCE_COMMIT = '8ad796e1bfa5d3f5dc697c3f14f62df0f24f0d29'
if SOURCE_COMMIT is None:
    raise RuntimeError('Pending source publication')
SOURCE_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
RUN_ID = 'inversion_crop_observation_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE = Path('/content') / (RUN_ID + '_source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', SOURCE_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', SOURCE_COMMIT], check=True)
if subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip() != SOURCE_COMMIT:
    raise RuntimeError('Source checkout differs from pinned SHA')
def version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
if version('torch') != '2.11.0+cu128':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run([sys.executable, '-c', "import torch,diffusers; assert str(torch.__version__) == '2.11.0+cu128', torch.__version__; assert diffusers.__version__ == '0.40.0', diffusers.__version__"], check=True)


In [ ]:
INPUT = Path('/content/drive/MyDrive/Video-WM/InversionStateHoldout/inversion_state_holdout_20260918T140243514638Z')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/InversionCropObservation') / RUN_ID
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ARCHIVE = OUTPUT.parent / (RUN_ID + '.source.zip')
subprocess.run(['git', '-C', str(SOURCE), 'archive', '--format=zip', '--output', str(ARCHIVE), SOURCE_COMMIT], check=True)
LOG = OUTPUT.parent / (RUN_ID + '.launcher.log')
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.inversion_crop_observation_run', '--input', str(INPUT), '--output', str(OUTPUT)]
with LOG.open('w') as log:
    process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        code = process.wait()
    except BaseException:
        try:
            os.killpg(process.pid, signal.SIGTERM); process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL); process.wait()
        except ProcessLookupError:
            pass
        raise
print('Output:', OUTPUT, 'Launcher log:', LOG, 'Source archive:', ARCHIVE)
if (OUTPUT / 'result.json').exists():
    result = json.loads((OUTPUT / 'result.json').read_text())
    compact = [{k:v for k,v in g.items() if k != 'per_video'} for g in result.get('oracle_summary',{}).get('groups',[])]
    print(json.dumps({'status':result['status'], 'clip_denominator':result['clip_denominator'], 'oracle_map_denominator':result['oracle_map_denominator'], 'fixed_calls':result['fixed_calls'], 'actual_calls_observed':result.get('actual_calls_observed'), 'oracle_summary':compact, 'claim':'all mappings oracle only; no best-phase selection or blind synchronization'}, indent=2))
if code:
    raise subprocess.CalledProcessError(code, command)
